In [0]:
%run ./init

This script is meant to be used as a cluster init script so that dependencies are installed automatically when the cluster starts. 

Use Tesseract OCR for extracting text from image or PDF documents.

Use Poppler (pdftoppm) to convert PDFs to images before OCR.

Hit:1 https://repos.azul.com/zulu/deb stable InRelease
Hit:2 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Reading package lists...


W: https://repos.azul.com/zulu/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.


Reading package lists...
Building dependency tree...
Reading state information...
poppler-utils is already the newest version (24.02.0-1ubuntu9.3).
tesseract-ocr is already the newest version (5.3.4-1build5).
0 upgraded, 0 newly installed, 0 to remove and 72 not upgraded.
/usr/bin/tesseract
/usr/bin/pdftoppm


path,name,size,modificationTime
dbfs:/databricks/init/install_poppler.sh,install_poppler.sh,142,1743112021000
dbfs:/databricks/init/install_system_dependencies.sh,install_system_dependencies.sh,203,1745335315000


...

/usr/bin/pdftoppm



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


/databricks/python/lib/python3.12/site-packages/huggingface_hub/file_download.py:832: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

'/dbfs/models/t5-small'

In [0]:
import os
import json
import logging

from PIL import Image
import pytesseract
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# ──────────────────────────────────────────────────────────────────────────────
# 0) GLOBAL LOGGING SETUP
# ──────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

# Silence low‑level chatter
for lib in ("py4j","py4j.java_gateway","pyspark","databricks","pdf2image"):
    logging.getLogger(lib).setLevel(logging.WARNING)

# Your logger—single handler, INFO only
logger = logging.getLogger("ImageKVExtractor")
logger.setLevel(logging.INFO)
logger.propagate = False
if not logger.handlers:
    h = logging.StreamHandler()
    h.setLevel(logging.INFO)
    h.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    logger.addHandler(h)


# ──────────────────────────────────────────────────────────────────────────────
# 1) IMAGE → TEXT → JSON extractor
# ──────────────────────────────────────────────────────────────────────────────
class ImageKVExtractor:
    """
    OCR each image in image_dir and extract fixed keys via a tiny T5 model.
    """

    KEYS = [
        "customer_name","contract_number","start_date","end_date",
        "iso","facility_meter_count","credit_fee","add_delete_fee",
        "add_delete_pct","mcv_fee","mcv_pct","bandwidth_premium"
    ]

    def __init__(self, image_dir: str, model_dir: str):
        """
        :param image_dir: directory with .png/.jpg files
        :param model_dir: local DBFS path to t5-small (or similar) offline
        """
        self.image_dir = image_dir

        logger.info(f"Loading T5‑Small from {model_dir}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=False)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_dir, local_files_only=False)
        device = 0 if torch.cuda.is_available() else -1
        self.pipe = pipeline(
            "text2text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device=device,
            max_length=512,
            do_sample=False
        )

    def list_images(self):
        imgs = [
            os.path.join(self.image_dir, f)
            for f in sorted(os.listdir(self.image_dir))
            if f.lower().endswith((".png","jpg","jpeg","tif"))
        ]
        logger.info(f"Found {len(imgs)} images in {self.image_dir}")
        return imgs

    def process_all(self) -> pd.DataFrame:
        records = []
        for img_path in self.list_images():
            filename = os.path.basename(img_path)
            logger.info(f"OCR + extract on {filename}")

            # 1) OCR
            text = pytesseract.image_to_string(Image.open(img_path))

            # 2) Prompt
            prompt = (
                "Extract these fields as JSON with keys: "
                + ", ".join(self.KEYS)
                + ". Null if missing.\n\n"
                + text
            )
            out = self.pipe(prompt)[0]["generated_text"]

            # 3) Parse
            try:
                data = json.loads(out)
            except json.JSONDecodeError:
                data = {"raw": out}

            data["file_name"] = filename
            records.append(data)

        df = pd.DataFrame(records)
        csv_out = os.path.join(self.image_dir, "extracted_fields.csv")
        df.to_csv(csv_out, index=False)
        logger.info(f"Results saved to {csv_out}")
        return df


# ──────────────────────────────────────────────────────────────────────────────
# 2) RUN ONCE
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    IMAGE_DIR = "/dbfs/mnt/mini-proj-dd/closed_deal_sheets"
    MODEL_DIR = "/dbfs/models/t5-small"   # or wherever you've placed your t5-small

    extractor = ImageKVExtractor(IMAGE_DIR, MODEL_DIR)
    df = extractor.process_all()
    print(df)


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
import os
import json
import logging

from PIL import Image
import pytesseract
import pandas as pd
from transformers import pipeline

# ──────────────────────────────────────────────────────────────────────────────
# GLOBAL LOGGING (INFO only)
# ──────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
for lib in ("py4j","pyspark","databricks","pdf2image"):
    logging.getLogger(lib).setLevel(logging.WARNING)

logger = logging.getLogger("ImageExtractor")
logger.setLevel(logging.INFO)
logger.propagate = False
if not logger.handlers:
    h = logging.StreamHandler()
    h.setLevel(logging.INFO)
    h.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    logger.addHandler(h)


# ──────────────────────────────────────────────────────────────────────────────
# QA‑BASED FIELD EXTRACTOR
# ──────────────────────────────────────────────────────────────────────────────
class ImageExtractor:
    """
    OCR each image file individually, then use DistilBERT QA
    to extract fixed-form fields into JSON.
    """
    QUESTIONS = {
        "customer_name":        "What is the customer name?",
        "contract_number":      "What is the contract ID or number?",
        "start_date":           "What is the start date?",
        "end_date":             "What is the end date?",
        "iso":                  "What ISO is listed?",
        "facility_meter_count": "What is the facility or meter count?",
        "credit_fee":           "What is the credit fee?",
        "add_delete_fee":       "What is the Add/Delete $ premium?",
        "add_delete_pct":       "What is the Add/Delete %?",
        "mcv_fee":              "What is the MCV fee or bandwidth premium?",
        "mcv_pct":              "What is the MCV %?"
    }

    def __init__(self):
        logger.info("Loading DistilBERT QA model (CPU)")
        self.qa = pipeline(
            "question-answering",
            model="distilbert-base-cased-distilled-squad",
            tokenizer="distilbert-base-cased-distilled-squad",
            device=-1
        )

    def _extract_from_text(self, text: str) -> dict:
        if not text.strip():
            # no text → all fields None
            return {k: None for k in self.QUESTIONS}
        out = {}
        for k, q in self.QUESTIONS.items():
            res = self.qa(question=q, context=text)
            out[k] = res["answer"] if res.get("score", 0) > 0.1 else None
        return out

    def process(self, images_root: str) -> pd.DataFrame:
        """
        Walk images_root recursively, OCR + QA each image file,
        output JSON/CSV and print JSON array.
        """
        records = []
        # find all images
        image_paths = []
        for root, _, files in os.walk(images_root):
            for f in files:
                if f.lower().endswith((".png","jpg","jpeg","tif")):
                    image_paths.append(os.path.join(root, f))
        logger.info(f"Found {len(image_paths)} image files under {images_root}")

        for img_path in sorted(image_paths):
            filename = os.path.relpath(img_path, images_root)  # keep subfolder/name
            logger.info(f"OCR + extract → {filename}")

            # OCR
            text = pytesseract.image_to_string(Image.open(img_path))
            print(text)
        


            # QA extraction
            fields = self._extract_from_text(text)
            fields["file_name"] = filename
            records.append(fields)

        # Save outputs
        json_path = os.path.join(images_root, "extracted_fields.json")
        csv_path  = os.path.join(images_root, "extracted_fields.csv")
        
        with open(json_path, "w") as jf:
            json.dump(records, jf, indent=2)
        pd.DataFrame(records).to_csv(csv_path, index=False)

        logger.info(f"✅ Saved JSON → {json_path}")
        logger.info(f"✅ Saved CSV  → {csv_path}")
        print(json.dumps(records, indent=2))

        return pd.DataFrame(records)


# ──────────────────────────────────────────────────────────────────────────────
# USAGE (run once)
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    IMAGES_ROOT = "/dbfs/mnt/mini-proj-dd/closed_deal_sheets"
    extractor = ImageExtractor()
    df = extractor.process(IMAGES_ROOT)


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

2025-04-21 23:48:55.997397: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-21 23:48:56.153089: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-21 23:48:56.332079: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745279336.482002    7493 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745279336.519694    7493 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-21 23:48:56.825345: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

...

Docusign Envelope ID: 6B3BBA1 F-7BF6-493A-A3A8-7BAE48A9D5A3

Shell Date Closed: 3/17/2025

Contract ID: TOY 3172025 112404
ENERG i Quote # 355885-6

CLOSED DEAL SHEET
ORIGINATOR: Scott Pease
PRICING ANALYST: Meagan

Musgrove

[CUSTOMER NAME: TOYO SOLAR TEXAS LLC

Deal Type: New
Contract Type: Commercial & Industrial
Contract Subtype:

KYC Approved: [X]

Product: ERCOT LMP Index _v09012024 Start Date: 4/1/2025 End Date: 4/1/2030 Term: 60

Rate/Adder: aed Facility Count: 3 Term MWh: 11,313.33

BROKER DETAILS

Broker: GETCHOICE! Broker Fee ($/MWh): LD Broker Rep:
QUOTE DETAILS

+
Payment Terms Late Fee %: HP MCV ho:

of.
Add/Delete > Renewable D> imbatance/PulCall 7:

E Adj ¢ Credit Fee:
nergy |ustments redit Fee @20)
PASS THROUGHS

Load Zone Basis,Energy Losses,RTRNA,REC,Ancillary,ISO Fees,CRR Adjustment,RUC,Default M,Uplift N,ERCOT Contingency Reserve
Service (ECRS),Firm Fuel Supply Service, RMR,

BILLING DETAILS Billing Sign Off
Bill Delivery: O Email O Print Tl Email & Print CO] Summa

2025-04-21 23:49:56,144 [INFO] OCR + extract → page_7.png


&) [Miley PJM CLOSED DEAL SHEET

A Shall Energy North America Subsidiary

Pricer: Babo Orgininator: Kendrick, Jo Anna Date Closed: _ 6/26/2018
l¥ New Deal ™ Renewal l™ Add ™ Other:
Utility: PenElec KYC Approved?

Customer Name: The Sisters of St. Joseph of NorthWestern Pennsylvania

Product: Fixed Price All-In Start Month: 10/2018 Term: 48 months

Rate: Gag’ Start Day: MRD Annual MWh: 1837

M special: == Revenue Class: Large

BROKER ™ Small
Broker: OnDemand Energy Bro Fee QD Broker Rep: .

QUOTE DETAILS

Payment Terms: 15 days Credit Fee: s6ie0 ! MVC: 25%
¥ other:
Add/Delete %: Renewable %: __ Imbalance/Put/Call: / /

Energy Adjustment: Reason:

PASS THROUGHS = (Cacti

™ Capacity T Energy Losses ™ RTRNA T™ ARR
™ Transmission ™ Ancillary = 1SO Fees
BILLING
T™ DUAL ¥ UCB Billing Sign off:
Bill Delivery: Email = Print Auto Pay [_]
i Fle -/E
Sales Tax Exempt f® Yes) & No Summary Billing [_]
L_ Sales Tax Documents Received? Distribution Groups [_]

Residential Use [_] Enrollment Sign off:



2025-04-21 23:50:03,151 [INFO] ✅ Saved JSON → /dbfs/mnt/mini-proj-dd/closed_deal_sheets/extracted_fields.json
2025-04-21 23:50:03,152 [INFO] ✅ Saved CSV  → /dbfs/mnt/mini-proj-dd/closed_deal_sheets/extracted_fields.csv


[
  {
    "customer_name": null,
    "contract_number": null,
    "start_date": "12/20/2018",
    "end_date": "12/20/2018",
    "iso": "Print Auto Pay",
    "facility_meter_count": null,
    "credit_fee": "$0.60",
    "add_delete_fee": null,
    "add_delete_pct": "Renewable %",
    "mcv_fee": null,
    "mcv_pct": null,
    "file_name": "page_1.png"
  },
  {
    "customer_name": null,
    "contract_number": "496D0FE6",
    "start_date": "2025",
    "end_date": "7/1/2026",
    "iso": "NYISO\nKYC Approved",
    "facility_meter_count": "Adder fie Facility",
    "credit_fee": null,
    "add_delete_fee": null,
    "add_delete_pct": "QP oy",
    "mcv_fee": null,
    "mcv_pct": "Add/Delete %: QP oy",
    "file_name": "page_10.png"
  },
  {
    "customer_name": "Ancillary I 1SO Fees\nBILLING",
    "contract_number": "\u20ac1 MiRRcy PJM",
    "start_date": "10/2018",
    "end_date": "5 2018",
    "iso": "Bill Delivery: Email \u2014 Print Auto Pay",
    "facility_meter_count": null,
    "credit_f

In [0]:
# import os
# import glob
# import logging

# import cv2
# import pandas as pd
# import pytesseract
# from pytesseract import Output

# # Configure root logger
# logging.basicConfig(
#     format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
#     level=logging.INFO
# )
# logger = logging.getLogger("ImageOCR")

# class ImageOCRExtractor:
#     """
#     OCR extractor for a folder of images.
#     - Lists all image files in the given DBFS folder.
#     - Applies preprocessing to each image.
#     - Runs Tesseract OCR and collects text + optional confidences.
#     - Returns a DataFrame and can write to CSV.
#     """
#     def __init__(self, folder_path: str, debug: bool = False):
#         """
#         :param folder_path: DBFS‐mounted path to images (e.g. '/dbfs/mnt/.../closed_deal_sheets')
#         :param debug: If True, will save intermediate preprocessed images and enable debug logging.
#         """
#         self.folder_path = folder_path
#         self.debug = debug
#         if self.debug:
#             logger.setLevel(logging.DEBUG)
#         logger.info(f"Initialized OCR extractor for folder: {folder_path}")

#     @staticmethod
#     def preprocess_image(img: cv2.Mat, debug: bool = False) -> cv2.Mat:
#         """
#         Preprocess the image to maximize OCR accuracy:
#         - Convert to gray
#         - Apply adaptive thresholding
#         You can extend this method with denoising, deskew, morphology, etc.
#         """
#         logger.debug(f"Preprocessing image of shape {img.shape}")
#         gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#         thresh = cv2.adaptiveThreshold(
#             gray, 255,
#             cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
#             cv2.THRESH_BINARY, 15, 9
#         )
#         if debug:
#             # Optionally save a copy for inspection
#             cv2.imwrite("/dbfs/tmp/preprocessed_debug.png", thresh)
#             logger.debug("Wrote debug preprocessed image to /dbfs/tmp/preprocessed_debug.png")
#         return thresh

#     def list_images(self) -> list:
#         """
#         List all common image files in the folder.
#         """
#         logger.info("Listing image files...")
#         patterns = ("*.png", "*.jpg", "*.jpeg", "*.tiff", "*.bmp")
#         files = []
#         for pat in patterns:
#             files.extend(glob.glob(os.path.join(self.folder_path, pat)))
#         files = sorted(files)
#         logger.info(f"Found {len(files)} image(s) in folder.")
#         return files

#     def extract(self) -> pd.DataFrame:
#         """
#         Process each image: load, preprocess, OCR, collect results.
#         Returns a DataFrame with columns:
#           - file_name, text, [optionally word, conf, left, top, width, height]
#         """
#         records = []
#         image_paths = self.list_images()
#         for img_path in image_paths:
#             fname = os.path.basename(img_path)
#             logger.info(f"Processing image: {fname}")

#             # 1) Read image
#             img = cv2.imread(img_path)
#             if img is None:
#                 logger.error(f"Failed to read image: {img_path}")
#                 continue

#             # 2) Preprocess
#             proc = self.preprocess_image(img, debug=self.debug)

#             # 3) OCR
#             logger.debug("Running Tesseract OCR...")
#             # If you only need full‑page text, use image_to_string
#             full_text = pytesseract.image_to_string(proc, lang="eng")
#             logger.debug("OCR complete for full text, length=%d chars", len(full_text))

#             # If you want word‑level data with confidences:
#             data = pytesseract.image_to_data(proc, output_type=Output.DATAFRAME)
#             data = data[data.conf.notna() & (data.conf >= 0)]
#             # And then you can iterate rows for detailed records.

#             # 4) Record result
#             records.append({
#                 "file_name": fname,
#                 "text": full_text
#             })

#         # Build DataFrame
#         df = pd.DataFrame(records)
#         logger.info("Extracted OCR text from all images. Total records: %d", len(df))
#         return df

#     def to_csv(self, df: pd.DataFrame, out_path: str):
#         """
#         Save the DataFrame to a CSV on DBFS.
#         :param df: DataFrame returned by extract()
#         :param out_path: Full DBFS path, e.g. '/dbfs/mnt/.../ocr_results.csv'
#         """
#         logger.info(f"Writing OCR results to CSV: {out_path}")
#         df.to_csv(out_path, index=False)
#         logger.info("CSV write complete.")

# # ---------------------------
# # Usage in a Databricks cell:
# # ---------------------------

# # 1) Initialize extractor as before
# extractor = ImageOCRExtractor(
#     folder_path="/dbfs/mnt/mini-proj-dd/closed_deal_sheets",
#     debug=False
# )

# # 2) Run extraction
# ocr_df = extractor.extract()

# # 3) Print out each file’s text directly
# for idx, row in ocr_df.iterrows():
#     print(f"\n--- File: {row['file_name']} ---\n")
#     print(row["text"])
#     print("\n" + "-"*50 + "\n")


# # 4) Save to CSV
# extractor.to_csv(
#     ocr_df,
#     out_path="/dbfs/mnt/mini-proj-dd/closed_deal_sheets/ocr_results.csv"
# )



--- File: page_1.png ---

@D |Mger PIM CLOSED DEAL SHEET

Contract 1D:17569-18055 |

Pricer: Tyler Orgininator: Kendrick, Jo Anna Date Closed: _ 12/20/2018
¥ New Deal IT Renewal rT Add ™ Other:
Utility: PECO KYC Approved?

Customer Name: 1600 Church Road Condominium Association

Product: Fixed Price All-In Start Month: 01/2019 Term: 36 months

Rate:_fs520) Start Day: MRD Annual MWh: 998

M special: Revenue Class: I Large

BROKER ¥ Small
Broker: Tobelmann Energ Bro Fee: Broker Rep:
QUOTE DETAILS .
Payment Terms: 15 days Credit Fee: MVC: _25%
v other(__|
Add/Delete %: Renewable %: Imbalance/Put/Call: / /

Energy Adjustment: Reason:

PASS THROUGHS —tcreckitpasethrel

r RPM Bc ™AC ™ RMR F’ Loss ™ TEC
™ NITS ™ TLe ™ RSV r™ RPS ™ ARR
BILLING
T DUAL 7 UCB Billing Sign off:
Bill Delivery: Email = Print Auto Pay [_]

Sales Tax Exempt Yes [7 No Summary Billing [_]

L.. Sales Tax Documents Received? Distribution Groups [7]

Residential Use [] Enrollment Sign off:

ENROLLMENT
[Meter Read Switch L

In [0]:
# import os, glob
# import logging

# import cv2
# import numpy as np
# import pytesseract
# from pytesseract import Output

# # configure logging
# logging.basicConfig(
#     format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
#     level=logging.INFO
# )
# logger = logging.getLogger("AlignedOCR")

# class AlignedOCR:
#     """
#     OCR that preserves word‑gaps by grouping on bounding boxes.
#     """
#     def __init__(self, folder: str, debug: bool=False):
#         self.folder = folder
#         self.debug = debug
#         if debug:
#             logger.setLevel(logging.DEBUG)
#         logger.info(f"AlignedOCR → watching folder {folder}")

#     @staticmethod
#     def preprocess_image(img, debug=False):
#         logger.debug("Preprocessing image for better contrast/thresholding")
#         gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#         thresh = cv2.adaptiveThreshold(
#             gray, 255,
#             cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
#             cv2.THRESH_BINARY, 15, 9
#         )
#         # Optionally save the preprocessed image for inspection:
#         if debug:
#             cv2.imwrite("/dbfs/tmp/ocr_debug.png", thresh)
#             logger.debug("Wrote /dbfs/tmp/ocr_debug.png")
#         return thresh

#     def list_images(self):
#         patterns = ["*.png","*.jpg","*.jpeg","*.tiff","*.bmp"]
#         files = []
#         for p in patterns:
#             files += glob.glob(os.path.join(self.folder, p))
#         files = sorted(files)
#         logger.info(f"Found {len(files)} images")
#         return files
#     def extract_and_print(self):
#         """
#         For each image:
#           - preprocess
#           - OCR to a DataFrame with bounding boxes
#           - group words into lines (unpacking 3 keys, not 4)
#           - rebuild each line with double‑spaces at large gaps
#           - print filename + text
#         """
#         files = self.list_images()
#         for path in files:
#             name = os.path.basename(path)
#             logger.info(f"→ Processing {name}")
#             img = cv2.imread(path)
#             if img is None:
#                 logger.error("  ❌ could not read image")
#                 continue

#             proc = self.preprocess_image(img, debug=self.debug)

#             config = r"--oem 3 --psm 6 -c preserve_interword_spaces=1"
#             df = pytesseract.image_to_data(proc, config=config, output_type=Output.DATAFRAME)

#             df = df[df.conf.notna() & (df.conf >= 0)]
#             if df.empty:
#                 logger.warning("  ⚠️ No OCR text detected")
#                 print(f"\n--- {name} (no text) ---\n")
#                 continue

#             # Correctly unpack 3 grouping keys: block_num, par_num, line_num
#             lines = []
#             for (block, par, line), group in df.groupby(["block_num","par_num","line_num"]):
#                 group = group.sort_values("left")
#                 lefts  = group["left"].tolist()
#                 widths = group["width"].tolist()
#                 words  = group["text"].tolist()
#                 # compute gap between words
#                 rights = [l + w for l, w in zip(lefts, widths)]
#                 gaps   = [ lefts[i] - rights[i-1] for i in range(1, len(lefts)) ]
#                 median_gap = float(np.median(gaps)) if gaps else 0.0

#                 # rebuild with extra spaces for large gaps
#                 line_text = words[0]
#                 for i, w in enumerate(words[1:], start=1):
#                     gap = gaps[i-1]
#                     if gap > median_gap * 5:
#                         line_text += "  " + w
#                     else:
#                         line_text += " " + w
#                 lines.append(line_text)

#             # print result
#             print(f"\n--- {name} ---\n")
#             print("\n".join(lines))
#             print("\n" + "-"*60 + "\n")


# # ------------------------------
# # Usage in your Databricks cell:
# # ------------------------------
# ocr = AlignedOCR("/dbfs/mnt/mini-proj-dd/closed_deal_sheets", debug=False)
# ocr.extract_and_print()



--- page_1.png ---

QW); [M2 ey PJM CLOSED DEAL SHEET [contract 1D:17569-18055
A Shall Cengy honk Amake Subddiory  ™  *
Pricer: Tyler  Orgininator: Kendrick, Jo Anna  Date Closed: _ 12/20/2018
iv New Deal  IT Renewal  i Add  ™ Other:
Utility: PECO  KYC Approved?
Customer Name: 1600 Church Road Condominium Association
Product: Fixed Price All-In  Start Month: 01/2019  Term: 36 months
Rate:_fs520)  Start Day: MRD  Annual MWh: 998
T special: nu. Revenue Class: I Large
BROAER ¥ Small
Broker: Tobelmann Energy  Bro Fee:  Broker Rep:
QUOTE DETAILS .
Payment Terms: 15 days  Credit Fee:  MVC: _25%
v other(__|
Add/Delete %: Renewable %: Imbalance/Put/Cal:_ /
Energy Adjustment: Reason:
tenet
nee nnn en
PASS THROUGHS  Icheck if pass-thrul
r RPM Pec ™ AC ™ RMR Loss ™ TEC
™ NITS  ™ TLe  ™ RSV  r™ RPS  ™ ARR
BILLING
tT DUAL  ¥ UCB  Billing Sign off:
Bill Delivery: Email = Print  Auto Pay [_]
Sales Tax Exempt Yes [7 No  Summary Billing [7]
L.. Sales Tax Documents Received?  Distribution Groups [7]
Re

In [0]:
# import os, glob
# import logging

# import cv2
# import numpy as np
# import pytesseract
# from pytesseract import Output

# # configure logging
# logging.basicConfig(
#     format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
#     level=logging.INFO
# )
# logger = logging.getLogger("AlignedOCR")

# class AlignedOCR:
#     """
#     OCR that preserves spatial gaps by classifying inter-word distances
#     into small, medium, and large, and inserting 1, 2, or 3 spaces.
#     """
#     def __init__(self, folder: str, debug: bool=False):
#         self.folder = folder
#         self.debug = debug
#         if debug:
#             logger.setLevel(logging.DEBUG)
#         logger.info(f"AlignedOCR → watching folder {folder}")

#     @staticmethod
#     def preprocess_image(img, debug=False):
#         gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#         thresh = cv2.adaptiveThreshold(
#             gray, 255,
#             cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
#             cv2.THRESH_BINARY, 15, 9
#         )
#         if debug:
#             cv2.imwrite("/dbfs/tmp/ocr_debug.png", thresh)
#             logger.debug("Wrote /dbfs/tmp/ocr_debug.png")
#         return thresh

#     def list_images(self):
#         patterns = ["*.png","*.jpg","*.jpeg","*.tiff","*.bmp"]
#         files = []
#         for p in patterns:
#             files += glob.glob(os.path.join(self.folder, p))
#         files = sorted(files)
#         logger.info(f"Found {len(files)} images")
#         return files

#     def extract_and_print(self):
#         """
#         For each image:
#           - preprocess
#           - OCR → DataFrame with bounding boxes
#           - group into lines
#           - compute per-line gap distribution
#           - set thresholds for small/medium/large gaps
#           - rebuild each line, inserting 1/2/3 spaces
#           - print with proper spacing
#         """
#         files = self.list_images()
#         for path in files:
#             name = os.path.basename(path)
#             logger.info(f"→ Processing {name}")
#             img = cv2.imread(path)
#             if img is None:
#                 logger.error(f"  ❌ could not read image: {path}")
#                 continue

#             proc = self.preprocess_image(img, debug=self.debug)
#             config = r"--oem 3 --psm 6 -c preserve_interword_spaces=1"
#             df = pytesseract.image_to_data(proc, config=config, output_type=Output.DATAFRAME)

#             df = df[df.conf.notna() & (df.conf >= 0)]
#             if df.empty:
#                 logger.warning(f"  ⚠️ No OCR text detected in {name}")
#                 print(f"\n--- {name} (no text) ---\n")
#                 continue

#             print(f"\n--- {name} ---\n")
#             # group by line
#             for (block, par, line_num), group in df.groupby(["block_num","par_num","line_num"]):
#                 group = group.sort_values("left")
#                 lefts  = group["left"].tolist()
#                 widths = group["width"].tolist()
#                 words  = group["text"].tolist()
#                 rights = [l + w for l, w in zip(lefts, widths)]
#                 # compute gaps between words
#                 gaps = [ lefts[i] - rights[i-1] for i in range(1, len(lefts)) ]
#                 if not gaps:
#                     # single-word line
#                     print(words[0])
#                     continue

#                 # determine thresholds:
#                 median_gap = float(np.median(gaps))
#                 q1 = np.percentile(gaps, 25)
#                 q3 = np.percentile(gaps, 75)
#                 iqr = q3 - q1
#                 # small_gap <= median_gap * 1.2
#                 # medium_gap if > small but <= (q3 + 0.5*iqr)
#                 large_gap = (gaps > (q3 + 0.5*iqr)).sum()
#                 small_thresh  = median_gap * 1.2
#                 medium_thresh = q3 + 0.5 * iqr

#                 # rebuild with 1/2/3 spaces
#                 line_text = words[0]
#                 for i, w in enumerate(words[1:], start=1):
#                     gap = gaps[i-1]
#                     if gap <= small_thresh:
#                         spacer = " "
#                     elif gap <= medium_thresh:
#                         spacer = "  "
#                     else:
#                         spacer = "   "
#                     line_text += spacer + w
#                 print(line_text)
#             print("\n" + "-"*60 + "\n")

# ocr = AlignedOCR("/dbfs/mnt/mini-proj-dd/closed_deal_sheets", debug=False)
# ocr.extract_and_print()



--- page_1.png ---

QW);   [M2  ey  PJM CLOSED DEAL SHEET  [contract 1D:17569-18055
A  Shall Cengy honk Amake Subddiory   ™   *
Pricer:  Tyler   Orgininator: Kendrick, Jo Anna   Date Closed:  _ 12/20/2018
iv New Deal  IT Renewal  i Add  ™ Other:
Utility: PECO   KYC Approved?
Customer Name: 1600 Church Road Condominium Association
Product: Fixed Price All-In   Start Month: 01/2019   Term:  36 months
Rate:_fs520)  Start Day:  MRD  Annual MWh: 998
T special:  nu.  Revenue Class:  I Large
BROAER  ¥ Small
Broker: Tobelmann Energy   Bro Fee:  Broker Rep:
QUOTE DETAILS  .
Payment Terms: 15 days   Credit Fee:   MVC: _25%
v other(__|
Add/Delete %: Renewable %:  Imbalance/Put/Cal:_  /
Energy Adjustment:  Reason:
tenet
nee  nnn en
PASS  THROUGHS   Icheck if pass-thrul
r RPM Pec ™ AC  ™ RMR Loss ™ TEC
™ NITS  ™ TLe  ™ RSV  r™ RPS  ™ ARR
BILLING
tT DUAL  ¥ UCB   Billing Sign off:
Bill Delivery:  Email  = Print   Auto Pay [_]
Sales Tax Exempt   Yes  [7 No   Summary Billing  [7]
L..  Sales Tax Docum

In [0]:
import os, glob, json, logging
import cv2
import numpy as np
import pytesseract
from pytesseract import Output

# ──────────────────────────────────────────────────────────────────────────────
# Logger
# ──────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    format="%(asctime)s [%(levelname)s] %(name)s ▶ %(message)s",
    level=logging.INFO
)
logger = logging.getLogger("KVExtractor")

class KeyValueOCR:
    """
    OCR + Key:Value extractor for scanned images.
    - Preserves multi‑space layout.
    - Splits on 2+ spaces to isolate segments.
    - Parses 'Key: Value' pairs.
    """
    def __init__(self, folder: str, debug: bool = False):
        """
        :param folder: DBFS‑mounted folder containing images.
        :param debug: If True, dumps preprocessed images to /dbfs/tmp for inspection.
        """
        self.folder = folder
        self.debug = debug
        if debug:
            logger.setLevel(logging.DEBUG)
        logger.info(f"Initialized KeyValueOCR for folder: {folder}")

    def list_images(self):
        exts = ["*.png","*.jpg","*.jpeg","*.tiff","*.bmp"]
        imgs = []
        for e in exts:
            imgs += glob.glob(os.path.join(self.folder, e))
        imgs = sorted(imgs)
        logger.info(f"Found {len(imgs)} images")
        return imgs

    @staticmethod
    def preprocess(img, debug=False):
        """
        Convert to grayscale + adaptive threshold for clearer OCR.
        """
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        thresh = cv2.adaptiveThreshold(
            gray, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, 15, 9
        )
        if debug:
            path = "/dbfs/tmp/kv_debug.png"
            cv2.imwrite(path, thresh)
            logger.debug(f"Wrote debug image to {path}")
        return thresh

    def extract_kv_from_line(self, line: str) -> dict:
        """
        Split a line on 2+ spaces, then parse each segment on first ':' into key/value.
        Returns a dict of any pairs found.
        """
        pairs = {}
        # split on two or more spaces
        segments = [seg.strip() for seg in re.split(r"\s{2,}", line) if seg.strip()]
        for seg in segments:
            if ":" in seg:
                key, val = seg.split(":", 1)
                key = key.strip()
                val = val.strip()
                if key and val:
                    pairs[key] = val
        return pairs

    def process(self) -> dict:
        """
        Loop through images, OCR + layout, then extract KV pairs.
        Returns a dict: { file_name: { key: value, ... }, ... }
        """
        import re

        results = {}
        for img_path in self.list_images():
            fname = os.path.basename(img_path)
            logger.info(f"Processing {fname}")
            img = cv2.imread(img_path)
            if img is None:
                logger.error(f"Could not read {fname}")
                continue

            proc = self.preprocess(img, debug=self.debug)
            cfg = "--oem 3 --psm 6 -c preserve_interword_spaces=1"
            df = pytesseract.image_to_data(proc, config=cfg, output_type=Output.DATAFRAME)
            df = df[df.conf.notna() & (df.conf >= 0)]
            if df.empty:
                logger.warning(f"No text detected in {fname}")
                results[fname] = {}
                continue

            # group into lines
            kvs = {}
            for (block_num, par_num, line_num), group in df.groupby(["block_num","par_num","line_num"]):
                group = group.sort_values("left")
                lefts  = group["left"].tolist()
                widths = group["width"].tolist()
                words  = group["text"].tolist()
                rights = [l+w for l,w in zip(lefts,widths)]
                gaps   = [ lefts[i] - rights[i-1] for i in range(1,len(lefts)) ]
                if not gaps:
                    line_text = words[0]
                else:
                    # dynamic thresholds
                    med = np.median(gaps)
                    q1, q3 = np.percentile(gaps, [25,75])
                    iqr = q3 - q1
                    small = med * 1.2
                    medium = q3 + 0.5*iqr
                    # rebuild
                    parts = [words[0]]
                    for i,w in enumerate(words[1:], start=1):
                        g = gaps[i-1]
                        if g <= small:
                            sp = " "
                        elif g <= medium:
                            sp = "  "
                        else:
                            sp = "   "
                        parts.append(sp + w)
                    line_text = "".join(parts)

                # extract KV from this line
                line_kv = self.extract_kv_from_line(line_text)
                kvs.update(line_kv)

            results[fname] = kvs
            logger.info(f"Extracted {len(kvs)} pairs from {fname}")

        return results

# ──────────────────────────────────────────────────────────────────────────────
# Usage
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    extractor = KeyValueOCR(
        folder="/dbfs/mnt/mini-proj-dd/closed_deal_sheets",
        debug=False
    )
    all_kvs = extractor.process()

    # 1) Print JSON to notebook
    print(json.dumps(all_kvs, indent=2))

    # 2) (Optional) Save it back to DBFS
    with open("/dbfs/mnt/mini-proj-dd/closed_deal_sheets/kv_output.json", "w") as f:
        json.dump(all_kvs, f, indent=2)


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
import os
import json
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.docstore.document import Document
from langchain.vectorstores import FAISS

# ── FUNCTIONS ─────────────────────────────────────────────────────────────────

def load_kv_json(path: str):
    """
    Load the JSON produced by your OCR key/value extractor and convert to LangChain Documents.
    """
    if not os.path.isfile(path):
        raise FileNotFoundError(f"JSON not found: {path!r}")
    with open(path, 'r') as f:
        raw = json.load(f)

    docs = []
    for fname, kv in raw.items():
        # join each key/value on a new line
        text = "\n".join(f"{k}: {v}" for k, v in kv.items())
        docs.append(Document(page_content=text, metadata={"source": fname}))
    return docs


def build_or_load_index(docs, embeddings, idx_path: str, rebuild: bool = False):
    """
    Build a new FAISS index or load an existing one from disk.
    """
    if rebuild or not os.path.isdir(idx_path):
        print(f"🔨 Building FAISS index at {idx_path} …")
        idx = FAISS.from_documents(docs, embeddings)
        idx.save_local(idx_path)
    else:
        print(f"📂 Loading FAISS index from {idx_path} …")
        # allow_dangerous_deserialization=True is required to load the pickle safely
        idx = FAISS.load_local(
            folder_path=idx_path,
            embeddings=embeddings,
            allow_dangerous_deserialization=True
        )
    return idx


if __name__ == '__main__':
    # ── CONFIG ─────────────────────────────────────────────────────────
    json_path   = '/dbfs/mnt/mini-proj-dd/closed_deal_sheets/kv_output.json'
    index_path  = '/dbfs/faiss_index'
    rebuild_idx = False  # set to True to force rebuilding the index

    # ── LOAD DOCUMENTS ────────────────────────────────────────────────
    docs = load_kv_json(json_path)

    # ── EMBEDDINGS & INDEX ────────────────────────────────────────────
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    index      = build_or_load_index(docs, embeddings, index_path, rebuild_idx)

    # ── RETRIEVER ─────────────────────────────────────────────────────
    # retrieve up to the total number of documents
    retriever = index.as_retriever(search_kwargs={'k': len(docs)})

    # ── QUERY FUNCTION ───────────────────────────────────────────────
    def query(q: str):
        """
        Retrieve top-k documents for q and print their source + contents.
        """
        results = retriever.get_relevant_documents(q)
        return '\n\n'.join(
            f"Source: {d.metadata['source']}\n{d.page_content}"
            for d in results
        )

    # ── EXAMPLE USAGE ─────────────────────────────────────────────────
    question = 'in page 12(1) what is the customer name?'
    print(f"Q: {question}\n")
    print(query(question))


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:434)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:464)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:737)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:508)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:613)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:636)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:49)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:293)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:62)
	at com.databricks.logging.AttributionContext$.withValue(Attr

In [0]:
import os
import json
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.docstore.document import Document
from langchain.vectorstores import FAISS

# ── FUNCTIONS ─────────────────────────────────────────────────────────────────

def load_kv_json(path: str):
    """
    Load the JSON produced by your OCR key/value extractor and convert to LangChain Documents.
    """
    if not os.path.isfile(path):
        raise FileNotFoundError(f"JSON not found: {path!r}")
    with open(path, 'r') as f:
        raw = json.load(f)

    docs = []
    for fname, kv in raw.items():
        # join each key/value on a new line
        text = "\n".join(f"{k}: {v}" for k, v in kv.items())
        docs.append(Document(page_content=text, metadata={"source": fname}))
    return docs


def build_or_load_index(docs, embeddings, idx_path: str, rebuild: bool = False):
    """
    Build a new FAISS index or load an existing one from disk.
    """
    if rebuild or not os.path.isdir(idx_path):
        print(f"🔨 Building FAISS index at {idx_path} …")
        idx = FAISS.from_documents(docs, embeddings)
        idx.save_local(idx_path)
    else:
        print(f"📂 Loading FAISS index from {idx_path} …")
        # allow_dangerous_deserialization=True is required to load the pickle safely
        idx = FAISS.load_local(
            folder_path=idx_path,
            embeddings=embeddings,
            allow_dangerous_deserialization=True
        )
    return idx


if __name__ == '__main__':
    # ── CONFIG ─────────────────────────────────────────────────────────
    json_path   = '/dbfs/mnt/mini-proj-dd/closed_deal_sheets/kv_output.json'
    index_path  = '/dbfs/faiss_index'
    rebuild_idx = False  # set to True to force rebuilding the index

    # ── LOAD DOCUMENTS ────────────────────────────────────────────────
    docs = load_kv_json(json_path)

    # ── EMBEDDINGS & INDEX ────────────────────────────────────────────
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    index      = build_or_load_index(docs, embeddings, index_path, rebuild_idx)

    # ── RETRIEVER ─────────────────────────────────────────────────────
    # retrieve up to the total number of documents
    retriever = index.as_retriever(search_kwargs={'k': len(docs)})

    # ── QUERY FUNCTION ───────────────────────────────────────────────
    def query(q: str):
        """
        Retrieve top-k documents for q and return their source + contents.
        """
        results = retriever.get_relevant_documents(q)
        return '\n\n'.join(
            f"Source: {d.metadata['source']}\n{d.page_content}"
            for d in results
        )

    # ── INTERACTIVE CHAT MODE ─────────────────────────────────────────
    print("Chatbot ready! 🎉 Type your question and press Enter (or 'exit' to quit).\n")
    while True:
        try:
            user_input = input("You: ")
        except (EOFError, KeyboardInterrupt):
            print("\nExiting. Goodbye! 👋")
            break
        if not user_input or user_input.strip().lower() in ('exit', 'quit'):
            print("Goodbye! 👋")
            break
        response = query(user_input)
        print(f"Bot: {response}\n")


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:434)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:464)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:737)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:508)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:613)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:636)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:49)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:293)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:62)
	at com.databricks.logging.AttributionContext$.withValue(Attr

In [0]:
# import os
# import json
# from langchain.embeddings import HuggingFaceEmbeddings
# from langchain.docstore.document import Document
# from langchain.vectorstores import FAISS
# from langchain.chains import RetrievalQA
# from langchain.llms import OpenAI

# # ── FUNCTIONS ─────────────────────────────────────────────────────────────────

# def load_kv_json(path: str):
#     """
#     Load the JSON produced by your OCR key/value extractor and convert to LangChain Documents.
#     """
#     if not os.path.isfile(path):
#         raise FileNotFoundError(f"JSON not found: {path!r}")
#     with open(path, 'r') as f:
#         raw = json.load(f)

#     docs = []
#     for fname, kv in raw.items():
#         # join each key/value on a new line
#         text = "\n".join(f"{k}: {v}" for k, v in kv.items())
#         docs.append(Document(page_content=text, metadata={"source": fname}))
#     return docs


# def build_or_load_index(docs, embeddings, idx_path: str, rebuild: bool = False):
#     """
#     Build a new FAISS index or load an existing one from disk.
#     """
#     if rebuild or not os.path.isdir(idx_path):
#         print(f"🔨 Building FAISS index at {idx_path} …")
#         idx = FAISS.from_documents(docs, embeddings)
#         idx.save_local(idx_path)
#     else:
#         print(f"📂 Loading FAISS index from {idx_path} …")
#         idx = FAISS.load_local(
#             folder_path=idx_path,
#             embeddings=embeddings,
#             allow_dangerous_deserialization=True
#         )
#     return idx

# if __name__ == '__main__':
#     # ── CONFIG ─────────────────────────────────────────────────────────
#     json_path   = '/dbfs/mnt/mini-proj-dd/closed_deal_sheets/kv_output.json'
#     index_path  = '/dbfs/faiss_index'
#     rebuild_idx = False  # set to True to force rebuilding the index

#     # ── LOAD DOCUMENTS ────────────────────────────────────────────────
#     docs = load_kv_json(json_path)

#     # ── EMBEDDINGS & INDEX ────────────────────────────────────────────
#     embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
#     index      = build_or_load_index(docs, embeddings, index_path, rebuild_idx)

#     # ── RETRIEVER ─────────────────────────────────────────────────────
#     # use top 3 for initial retrieval for focused context
#     retriever = index.as_retriever(search_kwargs={'k': 3})

#     # ── QA CHAIN SETUP ─────────────────────────────────────────────────
#     # instantiate an LLM (requires OPENAI_API_KEY in env)
#     llm = OpenAI(temperature=0)
#     qa_chain = RetrievalQA.from_chain_type(
#         llm=llm,
#         chain_type='stuff',  # or 'map_reduce' / 'refine' as needed
#         retriever=retriever,
#         return_source_documents=True
#     )

#     # ── INTERACTIVE CHAT MODE ─────────────────────────────────────────
#     print("Chatbot ready! 🎉 Ask questions about your deal sheets (type 'exit' to quit).\n")
#     while True:
#         try:
#             query_text = input("You: ")
#         except (EOFError, KeyboardInterrupt):
#             print("\nExiting. Goodbye! 👋")
#             break
#         if not query_text or query_text.strip().lower() in ('exit', 'quit'):
#             print("Goodbye! 👋")
#             break

#         # run through QA chain
#         result = qa_chain(query_text)
#         answer = result['result'] if isinstance(result, dict) else result
#         print(f"Bot: {answer}\n")

#         if isinstance(result, dict) and result.get('source_documents'):
#             print("--- Sources Used ---")
#             for doc in result['source_documents']:
#                 print(f"{doc.metadata['source']}")
#             print()

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:434)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:464)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:737)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:508)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:613)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:636)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:49)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:293)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:62)
	at com.databricks.logging.AttributionContext$.withValue(Attr

In [0]:
# import cv2
# import numpy as np
# import pytesseract
# from pytesseract import Output
# import glob, os, logging

# logging.basicConfig(level=logging.INFO)
# logger = logging.getLogger("AdvancedPreprocOCR")

# class AdvancedImageOCR:
#     def __init__(self, folder: str, debug: bool=False):
#         self.folder = folder
#         self.debug = debug
#         if debug:
#             logger.setLevel(logging.DEBUG)

#     def list_images(self):
#         exts = ["*.png","*.jpg","*.jpeg","*.tiff","*.bmp"]
#         files = []
#         for e in exts:
#             files += glob.glob(os.path.join(self.folder, e))
#         return sorted(files)

#     def deskew(self, img):
#         # Convert to binary for angle detection
#         gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#         thresh = cv2.threshold(gray, 0, 255,
#                                cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
#         coords = np.column_stack(np.where(thresh > 0))
#         angle = cv2.minAreaRect(coords)[-1]
#         if angle < -45:
#             angle = -(90 + angle)
#         else:
#             angle = -angle
#         logger.debug(f"Detected angle: {angle:.2f}")
#         (h, w) = img.shape[:2]
#         M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
#         return cv2.warpAffine(img, M, (w, h),
#                               flags=cv2.INTER_CUBIC,
#                               borderMode=cv2.BORDER_REPLICATE)

#     def clahe(self, img):
#         lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
#         l, a, b = cv2.split(lab)
#         clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
#         cl = clahe.apply(l)
#         return cv2.cvtColor(cv2.merge((cl,a,b)), cv2.COLOR_LAB2BGR)

#     def unsharp_mask(self, img):
#         blur = cv2.GaussianBlur(img, (0,0), sigmaX=3)
#         return cv2.addWeighted(img, 1.5, blur, -0.5, 0)

#     def denoise(self, img):
#         return cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)

#     def morph(self, img):
#         gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#         # binary inverse so text is white on black
#         thresh = cv2.adaptiveThreshold(
#             gray, 255,
#             cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
#             cv2.THRESH_BINARY_INV, 15, 9
#         )
#         kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2,2))
#         opened = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
#         closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel, iterations=1)
#         # invert back
#         return cv2.bitwise_not(closed)

#     def preprocess(self, img):
#         logger.debug("1) Deskewing")
#         img = self.deskew(img)
#         logger.debug("2) CLAHE contrast")
#         img = self.clahe(img)
#         logger.debug("3) Unsharp mask")
#         img = self.unsharp_mask(img)
#         logger.debug("4) Denoising")
#         img = self.denoise(img)
#         logger.debug("5) Morphological cleanup")
#         img = self.morph(img)
#         logger.debug("6) Rescaling x1.5")
#         img = cv2.resize(img, None, fx=1.5, fy=1.5,
#                          interpolation=cv2.INTER_CUBIC)
#         if self.debug:
#             path = "/dbfs/tmp/adv_preproc_debug.png"
#             cv2.imwrite(path, img)
#             logger.debug(f"Wrote debug image to {path}")
#         return img

#     def run(self):
#         cfg = r"--oem 3 --psm 6 -c preserve_interword_spaces=1"
#         for path in self.list_images():
#             name = os.path.basename(path)
#             logger.info(f"→ OCR on {name}")
#             img = cv2.imread(path)
#             proc = self.preprocess(img)
#             df = pytesseract.image_to_data(proc, config=cfg, output_type=Output.DATAFRAME)
#             df = df[df.conf.notna() & (df.conf>=0)]
#             text = pytesseract.image_to_string(proc, config=cfg)
#             logger.info(f"   OCR Tokens: {len(df)}, Text length: {len(text)}")
#             print(f"\n--- {name} ---\n{text}\n")

# # ──────────────────────────────────────────────────────────────────────────────
# # Usage in Databricks:
# # ──────────────────────────────────────────────────────────────────────────────
# if __name__ == "__main__":
#     ocr = AdvancedImageOCR(
#         folder="/dbfs/mnt/mini-proj-dd/closed_deal_sheets",
#         debug=True
#     )
#     ocr.run()


INFO:AdvancedPreprocOCR:→ OCR on page_1.png
DEBUG:AdvancedPreprocOCR:1) Deskewing
DEBUG:AdvancedPreprocOCR:Detected angle: -0.18
DEBUG:AdvancedPreprocOCR:2) CLAHE contrast
DEBUG:AdvancedPreprocOCR:3) Unsharp mask
DEBUG:AdvancedPreprocOCR:4) Denoising
DEBUG:AdvancedPreprocOCR:5) Morphological cleanup
DEBUG:AdvancedPreprocOCR:6) Rescaling x1.5
DEBUG:AdvancedPreprocOCR:Wrote debug image to /dbfs/tmp/adv_preproc_debug.png
INFO:AdvancedPreprocOCR:   OCR Tokens: 230, Text length: 2313
INFO:AdvancedPreprocOCR:→ OCR on page_10.png



--- page_1.png ---
cod aan           Poo,
“ALL:       AeA ares
yp        iM Be ey     PJM CLOSED DEAL SHEET [contract 1p:17569-18055
ee                            A Sicll Crergy Nowtk Armualca SoltkLony
Pricer: Tyler                    Orgininator: Kendrick. Jo Anna        Date Closed: 12/20/2018
7 New Deal           ™ Renewal          ™ Add          T Other:
Utility: PECO                                                                               KYC Approved?
Customer Name: 1600 Church Road Condominium Association
eee ee SOCatON
Product: Fixed Price All-In                    Start Month: 01/2019                       Term: 36 months
Rate:   [35.20                             Start Day: MRD                     Annual MWh: 998
special: Revenue Class: 7 Large
BROEER                                                                                         ¥ Small
Broker: Tobelmann Energy                         Bro Fee: Co Broker Rep:
QUOTE DETAILS                             .
Payment 

DEBUG:AdvancedPreprocOCR:1) Deskewing
DEBUG:AdvancedPreprocOCR:Detected angle: -0.72
DEBUG:AdvancedPreprocOCR:2) CLAHE contrast
DEBUG:AdvancedPreprocOCR:3) Unsharp mask
DEBUG:AdvancedPreprocOCR:4) Denoising
DEBUG:AdvancedPreprocOCR:5) Morphological cleanup
DEBUG:AdvancedPreprocOCR:6) Rescaling x1.5
DEBUG:AdvancedPreprocOCR:Wrote debug image to /dbfs/tmp/adv_preproc_debug.png
INFO:AdvancedPreprocOCR:   OCR Tokens: 254, Text length: 2458
INFO:AdvancedPreprocOCR:→ OCR on page_12 (1).png



--- page_10.png ---
Docusign Envelope ID: 496D0FE6-9563-4801-A715-2FFBA1D9CDE7
Shell                                                                 Date Closed: 1/29/2025
E VERGY                                        Contract ID: 100 1292025 51036
  IN     RSS                                                               Quote # 348786.2-6
ORIGINATOR: Charles Guajardo
PRICING ANALYST: Charles
Guajardo
CUSTOMER NAME: 100 FOREST AVE, LLC
Deal Type: New
Contract Type: Commercial & Industrial
Contract Subtype: Real Estate
SO: NYISO
KYC Approved: [X][ ]                                 Proxy Usage: [1] Yes [1 No
Product: _ Fixed Price                                   Start Date: 2/1/2025           End Date: 7/1/2026
Rate/Adder( J                                           Facility Count: 12                 Term: 17
erm MWh: 493MWh                                    Annual NWh: 348 MWh
BROKER DETAILS
Broker: Infinity Power Partners                        Broker Fee (MWh): 4            Brok

DEBUG:AdvancedPreprocOCR:1) Deskewing
DEBUG:AdvancedPreprocOCR:Detected angle: -90.00
DEBUG:AdvancedPreprocOCR:2) CLAHE contrast
DEBUG:AdvancedPreprocOCR:3) Unsharp mask
DEBUG:AdvancedPreprocOCR:4) Denoising
DEBUG:AdvancedPreprocOCR:5) Morphological cleanup
DEBUG:AdvancedPreprocOCR:6) Rescaling x1.5
DEBUG:AdvancedPreprocOCR:Wrote debug image to /dbfs/tmp/adv_preproc_debug.png
INFO:AdvancedPreprocOCR:   OCR Tokens: 502, Text length: 2232
INFO:AdvancedPreprocOCR:→ OCR on page_12.png



--- page_12 (1).png ---
6 OKlm@ ¢  41 8  3  2 ¢ 3828 $8 2 FP SF
rl   Fee [Tea ww GE     a    eg @  3 §  56 8 & 2 &
aun m of fg   = f      oO    =     =
moc WW O ct  + = Cc   da)  <   @  OSS   ° 3
pt oe ao 0  Ww   =      (D_ 5s =]  rR   ~  iu
wea t 24 9 » 2 ~ 2 “TOT ey  a (> + ff Oo fi    & @ =—
bom Gf  oF ro  @ re      QQ =a     =  3 <
2 5 |  moi 8m &  @ a0 8  2 8  op | a   k oF |m &
tn  i ou x § o   So  ct oe = fil ©   2 3 mb <

 = hi oS Ss   nv ©  3   nm sl 3   vO  S
———| BRR 88    3286  e   = [8   = 2  <

I g7 A gs  m   s< 6  o   he |2   o |Z  .

| a.  e( a 3   = oS     2 @ ID   2 lw  ou

    a |< &).   o @B     eT)  =   : 2

    alas =   3         ie   = a

     &  fa     a
cc |    ae         a   —<     bal

                       “i
.    Qo ~y 4 |       se        a
mm |  0  <@ FY &€       2        QO
> |  3  &§2 a4 9       a5        a  2
am  a  ~ 9         a        —  me

  9       tT 8    ®        S  ®
ms H  oh            4        D  4
= |  2       > m    -.        of  2
co |  5

DEBUG:AdvancedPreprocOCR:1) Deskewing
DEBUG:AdvancedPreprocOCR:Detected angle: -88.83
DEBUG:AdvancedPreprocOCR:2) CLAHE contrast
DEBUG:AdvancedPreprocOCR:3) Unsharp mask
DEBUG:AdvancedPreprocOCR:4) Denoising
DEBUG:AdvancedPreprocOCR:5) Morphological cleanup
DEBUG:AdvancedPreprocOCR:6) Rescaling x1.5
DEBUG:AdvancedPreprocOCR:Wrote debug image to /dbfs/tmp/adv_preproc_debug.png
INFO:AdvancedPreprocOCR:   OCR Tokens: 710, Text length: 5148
INFO:AdvancedPreprocOCR:→ OCR on page_19.png



--- page_12.png ---
Zz             eee ee .     . +                oy  . :   ..     4     1  . ot _    oO  oe
9   2 im       ‘29 2 oy  ee)       U     a      ;            Lott bois:      ;
a   S$ ‘3,  Mm    PIB mes   =e  ae  >     £  <'5 >   Dem gd: [= wv.    a     I> eB ip 2    Cc  iz |
2 &    s id        9\8:9-0 3   fc ipus &      2: SERB Biss 2:8 lg     >     ize. 3.8    HO ip 8
-    hoe 8 Bytes £ * 5 @     SSeS Ole S58: 8208    a    Belize gg Mi
zg       go oF    \5 a     sj      So 8 Big 18 iF a'S! [zi |e.    ed     gin, |B IB    oO
im   ae    aR 3”             T      8 see ) G28: 3:     5     ‘Ba: aie   = Oh
—      =       Fae      5                                  4  p ‘i ao   :    4 ‘3 Fs   lz   '          Oo          |   B » {8          rant
3s :      mm        |  Fs}      mH -                      8)                .   s    Ps   =  ran) sa     5:     ,        a           h & '       0        I
a an ti     :  2    / 8  i  Bh i   O  ed  =e
:    ~j    i]            .     c    

DEBUG:AdvancedPreprocOCR:1) Deskewing
DEBUG:AdvancedPreprocOCR:Detected angle: -89.20
DEBUG:AdvancedPreprocOCR:2) CLAHE contrast
DEBUG:AdvancedPreprocOCR:3) Unsharp mask
DEBUG:AdvancedPreprocOCR:4) Denoising
DEBUG:AdvancedPreprocOCR:5) Morphological cleanup
DEBUG:AdvancedPreprocOCR:6) Rescaling x1.5
DEBUG:AdvancedPreprocOCR:Wrote debug image to /dbfs/tmp/adv_preproc_debug.png


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
import os
import glob
import re
import json
import logging

import cv2
import numpy as np
import pytesseract
from pytesseract import Output

# ──────────────────────────────────────────────────────────────────────────────
# Logger Setup
# ──────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    format="%(asctime)s [%(levelname)s] %(name)s ▶ %(message)s",
    level=logging.INFO
)
logger = logging.getLogger("ContractDataExtractor")

class ContractDataExtractor:
    """
    1) Advanced preprocessing to “pop” text.
    2) OCR full page to a single string.
    3) Regex-based field extraction.
    """
    def __init__(self, folder: str, debug: bool = False):
        """
        :param folder: DBFS‑mounted path to your images folder.
        :param debug: save debug images and enable DEBUG logs.
        """
        self.folder = folder
        self.debug = debug
        if debug:
            logger.setLevel(logging.DEBUG)
        # Precompile common regex patterns
        self.patterns = {
            "Contract ID": re.compile(r"contract\s*(?:id)?\s*[:\-]\s*(?P<val>\S+)", re.IGNORECASE),
            "Date Closed": re.compile(r"date\s*closed\s*[:\-]\s*(?P<val>\d{1,2}/\d{1,2}/\d{4})", re.IGNORECASE),
            "Utility": re.compile(r"utility\s*[:\-]\s*(?P<val>[A-Za-z0-9 &]+)", re.IGNORECASE),
            "KYC Approved": re.compile(r"kyc\s*approved\?\s*(?P<val>\[X\]|\[ \]|\w+)", re.IGNORECASE),
            "Customer Name": re.compile(r"customer\s*name\s*[:\-]\s*(?P<val>.+?)(?=\n|\r|product\s*:)", re.IGNORECASE),
            "Product": re.compile(r"product\s*[:\-]\s*(?P<val>.+?)(?=\n|\r|start)", re.IGNORECASE),
            "Start Month": re.compile(r"start\s*month\s*[:\-]\s*(?P<val>\d{1,2}/\d{4})", re.IGNORECASE),
            "Start Date": re.compile(r"start\s*date\s*[:\-]\s*(?P<val>\d{1,2}/\d{1,2}/\d{4})", re.IGNORECASE),
            "End Date": re.compile(r"end\s*date\s*[:\-]\s*(?P<val>\d{1,2}/\d{1,2}/\d{4})", re.IGNORECASE),
            "Term": re.compile(r"term\s*[:\-]\s*(?P<val>\d+)\s*months", re.IGNORECASE),
            "Rate": re.compile(r"rate\s*[:\-]\s*(?P<val>[0-9\.\$%]+)", re.IGNORECASE),
            "Annual MWh": re.compile(r"annual\s*mwh\s*[:\-]\s*(?P<val>\d+)", re.IGNORECASE),
            "Broker": re.compile(r"broker\s*[:\-]\s*(?P<val>.+?)(?=\n|\r|broker\s*fee)", re.IGNORECASE),
            "Broker Fee": re.compile(r"broker\s*fee\s*(?:\(\S+\))?\s*[:\-]\s*(?P<val>[0-9\.\$%]+)", re.IGNORECASE),
            "Credit Fee": re.compile(r"credit\s*fee\s*[:\-]\s*(?P<val>[0-9\.\$%]+)", re.IGNORECASE),
            "MVC %": re.compile(r"MCV\s*%\s*[:\-]\s*(?P<val>[0-9\.%]+)", re.IGNORECASE),
            "Add/Delete %": re.compile(r"add\/delete\s*%\s*[:\-]\s*(?P<val>[0-9\.%]+)", re.IGNORECASE),
            "Renewable %": re.compile(r"renewable\s*%\s*[:\-]\s*(?P<val>[0-9\.%]+)", re.IGNORECASE),
        }

    def list_images(self):
        exts = ["*.png","*.jpg","*.jpeg","*.tiff","*.bmp"]
        imgs = []
        for e in exts:
            imgs.extend(glob.glob(os.path.join(self.folder, e)))
        imgs = sorted(imgs)
        logger.info(f"Found {len(imgs)} images")
        return imgs

    def preprocess(self, img: np.ndarray) -> np.ndarray:
        """
        Advanced preprocessing pipeline:
        1) Deskew
        2) CLAHE
        3) Unsharp mask
        4) Denoise
        5) Morphology
        6) Upscale
        """
        # 1) Deskew
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        thresh = cv2.threshold(gray, 0, 255,
                               cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)[1]
        coords = np.column_stack(np.where(thresh>0))
        angle = cv2.minAreaRect(coords)[-1]
        if angle < -45: angle = -(90 + angle)
        else:          angle = -angle
        (h, w) = img.shape[:2]
        M = cv2.getRotationMatrix2D((w//2,h//2), angle, 1)
        img = cv2.warpAffine(img, M, (w,h),
                             flags=cv2.INTER_CUBIC,
                             borderMode=cv2.BORDER_REPLICATE)
        logger.debug(f" Deskew angle: {angle:.2f}")

        # 2) CLAHE
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        img = cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2BGR)

        # 3) Unsharp
        blur = cv2.GaussianBlur(img, (0,0), sigmaX=3)
        img = cv2.addWeighted(img, 1.5, blur, -0.5, 0)

        # 4) Denoise
        img = cv2.fastNlMeansDenoisingColored(img, None, 10,10,7,21)

        # 5) Morphology
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        thresh = cv2.adaptiveThreshold(
            gray,255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV,15,9
        )
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT,(2,2))
        img = cv2.bitwise_not(cv2.morphologyEx(
            cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel), 
            cv2.MORPH_CLOSE, kernel
        ))

        # 6) Upscale
        img = cv2.resize(img, None, fx=1.5, fy=1.5,
                         interpolation=cv2.INTER_CUBIC)

        if self.debug:
            path = "/dbfs/tmp/adv_debug_preproc.png"
            cv2.imwrite(path, img)
            logger.debug(f"Saved debug preprocessed image to {path}")

        return img

    def ocr_full_text(self, img: np.ndarray) -> str:
        cfg = r"--oem 3 --psm 6 -c preserve_interword_spaces=1"
        text = pytesseract.image_to_string(img, config=cfg, lang="eng")
        logger.debug(f" OCR’d text length: {len(text)}")
        return text

    def extract_fields(self, text: str) -> dict:
        data = {}
        for name, pat in self.patterns.items():
            match = pat.search(text)
            if match:
                val = match.group("val").strip()
                data[name] = val
                logger.info(f"  Found {name}: {val}")
            else:
                logger.debug(f"  {name} not found")
        return data

    def process(self) -> dict:
        results = {}
        for img_path in self.list_images():
            fn = os.path.basename(img_path)
            logger.info(f"--- Processing {fn} ---")
            img = cv2.imread(img_path)
            if img is None:
                logger.error(f"Could not read {fn}")
                results[fn] = {}
                continue

            proc = self.preprocess(img)
            full_text = self.ocr_full_text(proc)
            # (Optional) print for inspection
            logger.debug(f"\n--- OCR Text for {fn} ---\n{full_text}\n")
            fields = self.extract_fields(full_text)
            results[fn] = fields
        return results

# ──────────────────────────────────────────────────────────────────────────────
# Usage in Databricks:
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    extractor = ContractDataExtractor(
        folder="/dbfs/mnt/mini-proj-dd/closed_deal_sheets",
        debug=True
    )
    output = extractor.process()

    # 1) Print JSON
    print(json.dumps(output, indent=2))

    # 2) (Optional) Save to DBFS
    with open("/dbfs/mnt/mini-proj-dd/closed_deal_sheets/contract_data.json", "w") as f:
        json.dump(output, f, indent=2)


INFO:ContractDataExtractor:Found 6 images
INFO:ContractDataExtractor:--- Processing page_1.png ---
DEBUG:ContractDataExtractor: Deskew angle: -0.18
DEBUG:ContractDataExtractor:Saved debug preprocessed image to /dbfs/tmp/adv_debug_preproc.png
DEBUG:ContractDataExtractor: OCR’d text length: 2182
DEBUG:ContractDataExtractor:
--- OCR Text for page_1.png ---
iaraioa           Lf
ALLA          wif t PRD
ay        iM Beer     PJM CLOSED DEAL SHEET [contract 1n:17569-18055
ee         A Sill Emegy Novk Amaka SobakLony
Pricer: Tyler                  Orgininator: Kendrick. Jo Anna       Date Closed: 12/20/2018
7 New Deal           I~ Renewal          ~ Add          T Other:
Utility: PECO                                                                               KYC Approved?
Customer Name: 1600 Church Road Condominium Association
eee eS OCattON
Product: Fixed Price All-In                    Start Month: 01/2019                       Term: 36 months
Rate:   £35.20                             St

{
  "page_1.png": {
    "Date Closed": "12/20/2018",
    "Utility": "PECO                                                                               KYC Approved",
    "KYC Approved": "Customer",
    "Customer Name": "1600 Church Road Condominium Association",
    "Product": "Fixed Price All-In",
    "Start Month": "01/2019",
    "Term": "36",
    "Annual MWh": "998",
    "Broker": "Tobelmann Energy                         Bro Fee: Broker Rep:"
  },
  "page_10.png": {
    "Contract ID": "100",
    "Date Closed": "1/29/2025",
    "Customer Name": "100 FOREST AVE, LLC",
    "Product": "_ Fixed Price",
    "Start Date": "2/1/2025",
    "End Date": "7/1/2026",
    "Rate": "10",
    "Annual MWh": "348",
    "Broker": "Infinity Power Partners",
    "Broker Fee": "4"
  },
  "page_12 (1).png": {},
  "page_12.png": {},
  "page_19.png": {},
  "page_7.png": {
    "Utility": "PenElec                                                                                                            KYC A